In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat, lit, floor, rand, explode, array

# Initialize Spark
spark = SparkSession.builder \
    .appName("SkewedDataLab") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()

# 1. Create a HEAVILY skewed Fact Table (e.g., Web Clicks)
# We will make 100,000 rows where 90% are 'GUEST' users
guest_rows = [("GUEST", "click_homepage") for _ in range(90000)]
normal_rows = [(f"user_{i}", "click_checkout") for i in range(10000)]
all_clicks = guest_rows + normal_rows

clickstream_df = spark.createDataFrame(all_clicks, ["user_id", "action"])

# 2. Create a Small Lookup Table (User Tiers)
user_tiers_data = [
    ("GUEST", "No Tier"),
    ("user_100", "Premium"),
    ("user_200", "Gold")
]
user_tiers_df = spark.createDataFrame(user_tiers_data, ["user_id", "tier"])

print("Data successfully generated!")

### big

In [0]:
display(clickstream_df.limit(5))

In [0]:
from pyspark.sql.functions import count

action_count_df = clickstream_df.groupBy("user_id").agg(count("action").alias("action_count"))
display(action_count_df.limit(5))


In [0]:
from pyspark.sql.functions import col, concat, lit, floor, rand, split, sum, when

SALT_FACTOR= 3

df_salted = clickstream_df.withColumn(
    "salted_user_id",
    when(col("user_id") == "GUEST", 
         concat(col("user_id"), lit("_"), floor(rand() * SALT_FACTOR)))
    .otherwise(col("user_id"))
)

stage1_df = df_salted.groupBy('salted_user_id').count()


stage2_df = stage1_df.withColumn(
    "original_user_id",
    when(col("salted_user_id").like("GUEST_%"), lit("GUEST"))
    .otherwise(col("salted_user_id"))
)



final_stage_df = stage2_df.groupBy('original_user_id').agg(sum('count').alias('final_action_count'))
display(final_stage_df.limit(10))

### small

In [0]:
display(user_tiers_df.limit(5))

In [0]:
from pyspark.sql.functions import broadcast

print("=== 1. STANDARD SHUFFLE JOIN PLAN ===")
standard_join = clickstream_df.join(user_tiers_df, "user_id")
# Look for 'SortMergeJoin' or 'ShuffleExchange' in the output below
standard_join.explain() 

print("\n" + "="*40 + "\n")

print("=== 2. OPTIMIZED BROADCAST JOIN PLAN ===")
broadcast_join = clickstream_df.join(broadcast(user_tiers_df), "user_id")
# Look for 'BroadcastHashJoin' and notice that 'ShuffleExchange' completely vanishes!
broadcast_join.explain()

In [0]:
print("=== FORCED STANDARD SHUFFLE JOIN PLAN (USING HINTS) ===")
SALT_FACTOR = 5

print("=== STEP 1: Salting the Skewed Fact Table ===")
salted_clickstream = clickstream_df.withColumn(
    "salted_user_id",
    concat(col("user_id"), lit("_"), floor(rand() * SALT_FACTOR))
)
salted_clickstream.show(5, truncate=False)


print("\n=== STEP 2: Exploding & Salting the Dimension Table ===")
# Create an array [0, 1, 2, 3, 4]
salt_array = array([lit(i) for i in range(SALT_FACTOR)])

exploded_tiers = user_tiers_df \
    .withColumn("salt", explode(salt_array)) \
    .withColumn("salted_user_id", concat(col("user_id"), lit("_"), col("salt"))) \
    .drop("salt")

exploded_tiers.show(10, truncate=False)


print("\n=== STEP 3: Executing the Salted Shuffle Join ===")
# Now we join on the newly created salted_user_id key
salted_final_join = salted_clickstream.join(exploded_tiers, on="salted_user_id", how="inner") \
                                      .drop("salted_user_id")

print(f"Final Count: {salted_final_join.count()} records processed smoothly!")
# We use .hint("merge") on the table to force Spark to execute a Sort-Merge Join
forced_shuffle_join = clickstream_df.join(user_tiers_df.hint("merge"), "user_id")

# This will bypass the automatic broadcast optimization and generate the shuffle plan
forced_shuffle_join.explain()